# Manhattan Starbucks: Hourly Demand Patterns

When does Manhattan need coffee most? This notebook analyzes **hourly subway ridership patterns** to understand time-of-day demand variation across Starbucks locations.

Using MTA turnstile data (Q4 2024, 4-hour granularity), we identify:
- **Peak vs off-peak demand hours** for each station
- **Morning-dominant vs evening-dominant** station clusters
- Implications for **staffing, hours of operation, and store format** (grab-and-go vs sit-down)

## Section 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

SB_GREEN = "#00704A"
SB_DARK  = "#1E3932"

ON_KAGGLE = Path("/kaggle/working").exists()

if ON_KAGGLE:
    _candidates = [
        Path("/kaggle/input/manhattan-cafe-wars"),
        Path("/kaggle/input/datasets/shiratoriseto/manhattan-cafe-wars"),
    ]
    DATA_DIR = next((p for p in _candidates if p.exists()), None)
    if DATA_DIR is None:
        raise FileNotFoundError("Dataset not found")
else:
    DATA_DIR = Path("../../dataset-upload/v3")

hourly = pd.read_csv(DATA_DIR / "mta_hourly_manhattan_q4_2024.csv")
stations = pd.read_csv(DATA_DIR / "manhattan_mta_ridership_summary.csv")
stores = pd.read_csv(DATA_DIR / "stores_enriched_v4.csv")

print(f"Hourly data: {hourly.shape[0]} rows ({hourly['station_complex_id'].nunique()} stations × {hourly['hour'].nunique()} hours)")
print(f"Stations: {stations.shape[0]}")
print(f"Stores: {stores.shape[0]}")

## Section 1 — Manhattan-Wide Hourly Profile

In [ ]:
# ── aggregate across all stations ────────────────────────────────────
hourly_agg = hourly.groupby("hour")["total_ridership"].sum().reset_index()
hourly_agg["pct"] = hourly_agg["total_ridership"] / hourly_agg["total_ridership"].sum() * 100

# Time labels
hour_labels = {0: "12am", 4: "4am", 8: "8am", 12: "12pm", 16: "4pm", 20: "8pm"}
hourly_agg["label"] = hourly_agg["hour"].map(hour_labels).fillna(hourly_agg["hour"].astype(str))

fig, ax = plt.subplots(figsize=(10, 5))
colors = [SB_GREEN if h in [8, 12, 16] else SB_DARK for h in hourly_agg["hour"]]
ax.bar(hourly_agg["hour"], hourly_agg["pct"], color=colors, edgecolor="white", width=3.5)
ax.set_xticks(hourly_agg["hour"])
ax.set_xticklabels(hourly_agg["label"])
ax.set_xlabel("Hour of Day")
ax.set_ylabel("% of Daily Ridership")
ax.set_title("Manhattan Subway Ridership by Hour — When Does the City Move?")

peak_hour = hourly_agg.loc[hourly_agg["pct"].idxmax()]
ax.annotate(f"Peak: {peak_hour['label']}\n({peak_hour['pct']:.1f}%)",
            xy=(peak_hour["hour"], peak_hour["pct"]),
            xytext=(peak_hour["hour"] + 3, peak_hour["pct"] + 1),
            arrowprops=dict(arrowstyle="->", color="red"),
            fontsize=10, color="red", fontweight="bold")

plt.tight_layout()
plt.show()

print("\nHourly breakdown:")
for _, row in hourly_agg.iterrows():
    bar = "█" * int(row["pct"] * 2)
    print(f"  {row['label']:>5s}: {row['pct']:5.1f}% {bar}")

## Section 2 — Station-Level Time Profiles

Different stations serve different functions — commuter hubs peak in mornings, entertainment districts peak at night.

In [ ]:
# ── compute morning vs evening ratio per station ──────────────────────
hourly_merged = hourly.merge(stations[["station_complex_id", "station_name", "lat", "lon"]],
                              on="station_complex_id", how="left")

# Morning = hours 4-12, Evening = hours 16-24
station_profiles = hourly_merged.groupby("station_complex_id").apply(
    lambda g: pd.Series({
        "station_name": g["station_name"].iloc[0],
        "lat": g["lat"].iloc[0],
        "lon": g["lon"].iloc[0],
        "morning": g.loc[g["hour"].isin([4, 8]), "total_ridership"].sum(),
        "midday": g.loc[g["hour"].isin([12]), "total_ridership"].sum(),
        "evening": g.loc[g["hour"].isin([16, 20]), "total_ridership"].sum(),
        "night": g.loc[g["hour"].isin([0]), "total_ridership"].sum(),
        "total": g["total_ridership"].sum(),
    })
).reset_index()

station_profiles["morning_pct"] = station_profiles["morning"] / station_profiles["total"] * 100
station_profiles["evening_pct"] = station_profiles["evening"] / station_profiles["total"] * 100
station_profiles["am_pm_ratio"] = station_profiles["morning"] / (station_profiles["evening"] + 1)

# Top morning-heavy and evening-heavy stations
print("Top 5 Morning-Dominant Stations (commuter inflow):")
for _, row in station_profiles.nlargest(5, "am_pm_ratio").iterrows():
    name = row["station_name"][:50]
    print(f"  {name:<50s} AM/PM ratio: {row['am_pm_ratio']:.2f}  Morning: {row['morning_pct']:.0f}%")

print(f"\nTop 5 Evening-Dominant Stations (entertainment/nightlife):")
for _, row in station_profiles.nsmallest(5, "am_pm_ratio").iterrows():
    name = row["station_name"][:50]
    print(f"  {name:<50s} AM/PM ratio: {row['am_pm_ratio']:.2f}  Evening: {row['evening_pct']:.0f}%")

In [ ]:
# ── map: morning vs evening stations ──────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 12))

sp = station_profiles.dropna(subset=["lat", "lon"])
sc = ax.scatter(sp["lon"], sp["lat"],
                c=sp["am_pm_ratio"], cmap="RdYlBu",
                s=sp["total"] / sp["total"].max() * 200 + 20,
                edgecolors="grey", linewidth=0.3, alpha=0.8,
                vmin=0.5, vmax=2.0)
plt.colorbar(sc, ax=ax, label="AM/PM Ratio (>1 = morning-heavy)", shrink=0.7)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Station Demand Profiles — Blue = Morning, Red = Evening")
plt.tight_layout()
plt.show()

## Section 3 — Starbucks × Time-of-Day: Store Format Implications

In [ ]:
# ── link stores to nearest station profiles ──────────────────────────
from scipy.spatial.distance import cdist

store_coords = stores[["lat", "lon"]].dropna().values
station_coords = sp[["lat", "lon"]].values

dists = cdist(store_coords, station_coords)
nearest_idx = np.argmin(dists, axis=1)

stores_with_profile = stores.dropna(subset=["lat", "lon"]).copy()
stores_with_profile["nearest_station"] = sp.iloc[nearest_idx]["station_name"].values
stores_with_profile["am_pm_ratio"] = sp.iloc[nearest_idx]["am_pm_ratio"].values
stores_with_profile["morning_pct"] = sp.iloc[nearest_idx]["morning_pct"].values

# Classify stores
stores_with_profile["store_type"] = pd.cut(
    stores_with_profile["am_pm_ratio"],
    bins=[0, 0.8, 1.2, 999],
    labels=["Evening Hub", "Balanced", "Morning Commuter"]
)

type_counts = stores_with_profile["store_type"].value_counts()
print("Store Classification by Nearest Station Profile:")
print(f"  Morning Commuter (grab-and-go priority): {type_counts.get('Morning Commuter', 0)} stores")
print(f"  Balanced (all-day format):                {type_counts.get('Balanced', 0)} stores")
print(f"  Evening Hub (extended hours viable):      {type_counts.get('Evening Hub', 0)} stores")

# ── bar chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colors = {"Morning Commuter": "#1976D2", "Balanced": SB_GREEN, "Evening Hub": "#D32F2F"}
type_counts.plot(kind="barh", ax=ax, color=[colors.get(t, "grey") for t in type_counts.index])
ax.set_xlabel("Number of Stores")
ax.set_title("Starbucks Store Types by Demand Profile")
plt.tight_layout()
plt.show()

## Section 4 — Key Findings

### 1. Manhattan demand is bimodal
Morning (8am) and evening (4–8pm) peaks dominate ridership. Starbucks stores near commuter-inflow stations should prioritize **speed and throughput** during the 7–9am window.

### 2. Evening-dominant stations are underserved
Stations near entertainment districts show high evening ridership but Starbucks typically closes by 8–9pm. **Extended hours** at these locations could capture after-dinner and theater traffic.

### 3. Store format should match demand profile
- **Morning Commuter stores** → optimize for grab-and-go: mobile order pickup, minimal seating
- **Balanced stores** → full-service with seating, food menu expansion
- **Evening Hub stores** → consider alcohol/evening menu, later closing times

### Generalization
This hourly demand analysis framework applies to any location-based service. Swap MTA data for transit data in your city, and the same morning/evening classification reveals optimal service design.

## Section 5 — Limitations & Series Navigation

### Limitations

| # | Limitation | Impact |
|---|-----------|--------|
| 1 | **4-hour granularity** — MTA data is bucketed into 4h windows | Cannot pinpoint exact peak hour (e.g., 8am vs 9am) |
| 2 | **Q4 2024 only** — one quarter of data | Seasonal variation (summer tourism) not captured |
| 3 | **Subway ≠ all foot traffic** — pedestrians, taxis, buses not included | Some stores may serve primarily non-transit customers |
| 4 | **No actual Starbucks sales data** — ridership is a demand proxy | High ridership doesn't guarantee high coffee demand at all hours |

### Series Navigation

| # | Notebook | Link |
|---|----------|------|
| 0 | Manhattan Café Wars — EDA | [Kaggle](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) |
| 1 | Starbucks 10-K NLP: Topic & Keyword Analysis | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) |
| 1F | Starbucks 10-K LDA Topic Explorer (pyLDAvis) | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-lda-topic-explorer-pyldavis) |
| 2A | Starbucks Spatial Clustering | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) |
| 2B | Starbucks Location Fitness | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) |
| 2C | Starbucks Walk-Distance Analysis (OSMnx) | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-walk-distance-analysis-osmnx) |
| 5A | LFS Predictive Model | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness-score-predictive-model) |
| 5B | Strategic Location Insights | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-strategic-location-insights) |
| 5C | Optimal Store Placement | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-optimal-store-placement) |
| **5D** | **Hourly Demand Patterns (this notebook)** | — |
| C | Data Pipeline: EDGAR + OSM → CSV | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-data-pipeline-edgar-osm-to-csv) |
| D | US Expansion Animated Choropleth | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-us-expansion-animated-choropleth) |
| G | NLP × Spatial Integration | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-nlp-x-spatial) |

**Dataset:** [shiratoriseto/manhattan-cafe-wars](https://www.kaggle.com/datasets/shiratoriseto/manhattan-cafe-wars)  
**GitHub:** [seto-siratori/starbucks-kaggle](https://github.com/seto-siratori/starbucks-kaggle)